## tl;dr

V32 frozen pre-K1 volume-wave gate did not pass. The 100 accepted 4h marks average -18.445021 bp after the fixed 20 bp cost threshold. This is not executable profit.

## Context & Methods

Companion to the Chinese HTML report. Saved 2023–2024 only; no raw/2025+/holdout. Original 251 mothers/744 controls,100 accepted compare ALL300 original controls. Four-hour primary,20bp unchanged; other horizons descriptive.

### Key Assumptions

This reused development sample is exploratory. Monthly cluster inference assumes weak between-month dependence; it is not randomized treatment. This notebook reads only the saved ledgers and independent audit. Run from the repository or any descendant with system Python NumPy2.0.2 / pandas2.3.3. No new results are written.

In [ ]:
from pathlib import Path
import csv, gzip, json, math, hashlib, importlib.util
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"yoyo/evaluation/hourly_impulse_volume_wave_economics.py").exists())
directory = root/"experiments/active/exp-btcusdtp-1h-volume-wave-economics-preholdout-20260907-v32"
summary = json.loads((directory/"results/summary.json").read_text())
def rows(name):
    with gzip.open(directory/"results"/name, "rt") as stream:
        return list(csv.DictReader(stream))
cases = rows("case_ledger.csv.gz")
mothers = rows("mother_ledger.csv.gz")
print("Saved label rows:", len(cases), "mother rows:", len(mothers))

## Data

Profile the complete event×horizon grid and reconcile output hashes. Empty policy cells reflect unknown gates, not zero-profit trades.

In [ ]:
for name, expected in summary["output_hashes"].items():
    assert hashlib.sha256((directory/"results"/name).read_bytes()).hexdigest() == expected
for name, records in (("cases", cases), ("mothers", mothers)):
    keys = {(r["event_id"],r["horizon_hours"]) for r in records}
    assert len(keys) == len(records) == 1004
    print(name, "unique keys:", len(keys), "own-decision range:", min(r["decision_time"] for r in records), max(r["decision_time"] for r in records))

## Results

Recompute failure counts and same-opportunity cost decomposition from the saved ledger. Unknown gate remains excluded from the policy mean and explicitly counted.

In [ ]:
four = [r for r in cases if r["horizon_hours"] == "4"]
accepted = [r for r in four if r["wave_gate_state"] == "accepted"]
abstained = [r for r in four if r["wave_gate_state"] == "abstain"]
unknown = [r for r in four if r["wave_gate_state"] == "unknown"]
assert (len(accepted),len(abstained),len(unknown)) == (100,148,3)
mean = lambda xs: math.fsum(xs)/len(xs)
net = [float(r["cost_threshold_markout"]) for r in accepted]
gross = [float(r["gross_markout"]) for r in accepted]
print("Accepted gross/net bp:", mean(gross)*1e4, mean(net)*1e4)
print("Wrong direction / cost-erased / net-positive:",sum(v<0 for v in gross),sum(0<v<=.002 for v in gross),sum(v>0 for v in net))
cost = len(abstained)*20/248
gross_change = -math.fsum(float(r["gross_markout"]) for r in abstained)*1e4/248
print("Avoided costs / net gross contribution / policy improvement bp:",cost,gross_change,cost+gross_change)
for fold in sorted({r["fold"] for r in four}):
    group=[float(r["cost_threshold_markout"]) for r in accepted if r["fold"]==fold]
    print(fold,len(group),mean(group)*1e4)
assert summary["decision"]["status"]=="not_supported"

### Independent saved-number replay

The auditor does not import the strategy core. It independently reconstructs saved endpoint arithmetic and monthly statistics; it does not acquire market prices. check-only does not overwrite its saved receipt.

In [ ]:
spec = importlib.util.spec_from_file_location("_notebook_v32_audit",root/"scripts/audit_hourly_impulse_volume_wave_economics_v32.py")
auditor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(auditor)
audit = auditor.run(root,write_receipt=False)
print({k:audit[k] for k in ("status","label_rows_recomputed","mother_horizon_rows_recomputed","primary_inferences_recomputed","raw_prices_read","holdout_consumed")})

## Takeaways

The retained events do not cover20bp. Policy improvement comes from avoiding fees; discarded gross returns were positive in aggregate. Do not use the slight long-side mean, a different horizon or reduced costs to reclassify this frozen result. Endpoint failure categories do not establish stop-loss path, consolidation or fake-breakout causes.

Execution record: pure-Python code cells are replayed sequentially by the documented standard-library runner because no Jupyter/nbclient/nbformat is installed. This validates code cells and captures actual outputs, not Jupyter UI/kernel compatibility.